## 6. 卷积神经网络

- 在第三章和第四章的内容中，我们处理了FashionMnist这个数据集，但当时无论是单层的线性、非线性神经网络都未能很好地捕捉图像真正的先验信息和特征，联想到图像这一信息载体，很显然的一个特征就是相邻像素之间的相互关联性，利用这部分先验知识往往能获得更好的空间结构信息。

- 本章就会介绍一类几乎可以说是针对于图像数据而设计的网络，即卷积神经网络(convolutional neural network,CNN),围绕图像展开的一系列计算机视觉任务，如图像识别、目标检测或语义分割等任务都常常以这类网络为基础。

- 卷积的操作非常适合在GPU上并行计算，除了上面提到的图像数据，卷积神经网络也常常用于序列结构的任务、图结构数据和推荐系统中。

### 6.1 从全连接层到卷积

- MLP在面对一些任务时可能会比较棘手：例如有一张百万级像素的图像，即使我们设置隐藏层的维度是1000，单层的全连接层的weight数量也将达到十亿级别，这几乎和现代的小型LLM达到一个级别，但如此庞大的单层数据规模可能根本无法学习到良好的图像特征，在同等的参数规模下，其他的网络结构能捕获更大范围、更多层次、更多类别的信息，因此，我们迫切的需要利用当前数据本身的特性，CNN就是这类创造性方法的一种。

#### 6.1.1 不变性

- 围绕上面所说的内容，我们应该去思考图像数据中存在什么样的先验结构信息：
    - 我们得到的结论非常简单也非常直观，即**如果我们试图完成一个从图像中寻找某个特定物体的任务，我们选择的方法一定要和物体的位置无关**，例如，我们可以将图像分割成多块，按照相同的方式类似递归的去寻找这个物体，而在递归函数中，寻找这个行为一定是相同的，换言之，在图像的哪个位置我们去寻找都需要一样的方式方法
    - 我们总结上面的内容：即**如果要去寻找一种适合图像数据的网络结构设计方法，要么一定要满足平移不变性，即不管检测对象出现在图像哪里，网络都需要对这个位置做出相同的反应**
    - 后面的局部性我认为用识别这个任务说有点牵强，我换种说法：图像数据本身的很多浅显的信息就集中在局部，例如我们如果看到一张图像中有几朵云，那么大概率离这里很近的位置就会有一个太阳or月亮，这是能在不用看完整张图像就能知道的现象；但如果我问你这张图像表达了拍摄者什么样的构思，你就不能只用局部来回答我了，你得看完整张图像并对图像中展示的画面、意境、构图有一个全面的认知，也即这里的**网络前期不应过度在意较远区域的关系，最终我们能聚合这些局部特征，对整个图像级别进行预测**

#### 6.1.2 MLP的限制